# Task 3 — frozen GNN–BERT fusion

Run the five ablations on paired embeddings. Synthetic mode verifies execution only; its scores are not research results. See `docs/task3/README.md` for aligned real-data preparation.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.task3.data import make_demo
from src.task3.train import run_experiments
import pandas as pd
import json
from IPython.display import Image, display

## Configuration

Use a fresh output directory each time. Real mode needs frozen features prepared from encoders trained with aligned splits. Evaluation defaults to validation; enable test only after fixing your experiment.

In [ ]:
from datetime import datetime
USE_SYNTHETIC = True
EVALUATE_TEST = False
PATIENCE = 5
LEARNING_RATE = 1e-3
FEATURES = ROOT / 'data/processed/task3/features.pt'
OUTPUT = ROOT / ('tmp/task3_notebook' if USE_SYNTHETIC else 'results/task3') / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
if USE_SYNTHETIC:
    FEATURES = make_demo(OUTPUT.with_suffix('.features.pt'))
print('Mode:', 'synthetic diagnostics' if USE_SYNTHETIC else 'real paired data')

In [ ]:
report = run_experiments(FEATURES, OUTPUT, epochs=2 if USE_SYNTHETIC else 20,
                         device='auto', evaluate_test=EVALUATE_TEST,
                         patience=PATIENCE, learning_rate=LEARNING_RATE)
display(pd.read_csv(OUTPUT / 'comparison.csv'))
print(json.dumps(report, indent=2))

In [ ]:
display(Image(filename=str(OUTPUT / 'validation_curves.png')))
if (OUTPUT / 'embeddings.png').exists():
    display(Image(filename=str(OUTPUT / 'embeddings.png')))
display(pd.DataFrame(json.loads((OUTPUT / 'cases.json').read_text(encoding='utf-8'))))

## Evaluate the saved winner without retraining
Enable only after fixing the experiment. To evaluate a previous run, set OUTPUT and FEATURES to its paths and run this cell without rerunning training.

In [ ]:
RUN_SAVED_EVALUATION = False
if RUN_SAVED_EVALUATION:
    from src.task3.evaluate import evaluate_checkpoint
    selection = json.loads((OUTPUT / 'selection.json').read_text())
    saved_report = evaluate_checkpoint(OUTPUT / selection['checkpoint'], FEATURES, OUTPUT / 'final_test', device='auto')
    print(json.dumps(saved_report, indent=2))

## Interpretation

Compare validation scores before inspecting test. The unimodal controls use the same frozen features and samples as fusion. Check three saved cases for mistakes. t-SNE is descriptive and attention is not a causal explanation. Caption-derived labels remain a leakage limitation; synthetic fixtures cannot demonstrate real music understanding.